# QML Trojan Benchmark — demonstração didática

Este notebook percorre, em pequena escala, as quatro perguntas do projeto:

1. treinar um **classificador quântico variacional (VQC)** limpo (*golden*);
2. inserir um **trojan de porta única** via um passe de transpilação malicioso e medir o desvio da distribuição de saída (**TVD**) e o impacto na **acurácia**;
3. construir um **backdoor com gatilho** (*conditional trojan*) e medir a **taxa de sucesso do ataque (ASR)** contra a acurácia limpa;
4. testar a **detecção textual** (QASM como texto) de circuitos limpos vs. infectados.

> Para os experimentos completos e reprodutíveis, use os scripts em `scripts/` (`python scripts/run_all.py --quick`). Aqui o objetivo é *entender o mecanismo*.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

from qmltrojan.data import load_dataset
from qmltrojan.vqc import VQCSpec, train_vqc, compile_golden, accuracy
from qmltrojan.trojan import TrojanSpec, apply_trojan, BackdoorSpec, build_backdoor, make_trigger_inputs
from qmltrojan.sensitivity import distribution_metrics, predict_sampled, sampled_accuracy
from qmltrojan.io_qasm import to_qasm3

np.set_printoptions(precision=3, suppress=True)

## 1. Treinar um VQC limpo (*golden*)

Codificamos cada amostra em ângulos de rotação (um qubit por *feature*, via PCA), aplicamos um *ansatz* `RealAmplitudes` e lemos a **expectativa da paridade** `⟨Z⊗Z⊗…⟩`. A classe prevista é o sinal dessa expectativa.

In [ ]:
spec = VQCSpec(dataset="breast_cancer", encoding="z", ansatz_reps=2, n_qubits=4, seed=0)
data = load_dataset(spec.dataset, spec.n_qubits, spec.seed, max_train=150)
trained = train_vqc(spec, data, maxiter=120)
golden = compile_golden(spec, trained.weights)
print(f"Acuracia limpa  treino={trained.train_accuracy:.3f}  teste={trained.test_accuracy:.3f}")

Um trecho do circuito treinado, já em **OpenQASM 3** (as *features* ficam como `input float[64]`):

In [ ]:
for line in to_qasm3(golden).splitlines()[:14]:
    print(line)

## 2. Inserir um trojan de porta única

Um passe de transpilação comprometido insere **uma** porta. Comparamos portas diagonais (`Z`, `T`) e não-diagonais (`X`, `Y`) inseridas na fronteira codificação/ansatz.

In [ ]:
X_eval = data.X_test[:8]
for gate in ["z", "x", "y", "t"]:
    infected, rec = apply_trojan(golden, TrojanSpec(gate=gate, mode="controlled", seed=0))
    met = distribution_metrics(golden, infected, X_eval, shots=512, reps=10, seed=0)
    acc = accuracy(infected, data.X_test, data.y_test)
    print(f"{gate.upper()}: TVD={met['tvd_mean']:.3f}+/-{met['tvd_std']:.3f} "
          f"(baseline {met['tvd_baseline_mean']:.3f})  acuracia={acc:.3f} "
          f"(limpa {trained.test_accuracy:.3f})")

O `baseline` é o TVD *golden × golden* (ruído de amostragem). Só valores **acima** dele indicam efeito real do trojan. Portas diagonais tendem a ter impacto menor na classe quando próximas da medição; `X`/`Y` invertem a paridade.

## 3. Backdoor com gatilho (*conditional trojan*)

O gatilho é "as primeiras *k* *features* no topo da faixa de codificação (π/2)". Um detector constante (inverso da codificação no ângulo do gatilho) marca uma **ancila**, medida no meio do circuito; um `X` classicamente controlado inverte a classe **apenas** quando o gatilho dispara.

In [ ]:
n_data = spec.n_qubits
for k in [1, 2, 3]:
    bd, gold, _ = build_backdoor(spec, trained.weights, BackdoorSpec(k_trigger=k))
    clean = sampled_accuracy(bd, data.X_test, data.y_test, n_data, shots=2048, seed=0)
    X_trig = make_trigger_inputs(n_data, k, np.pi/2, 60, base=data.X_test, seed=0)
    asr = np.mean(predict_sampled(gold, X_trig, n_data) != predict_sampled(bd, X_trig, n_data))
    print(f"k={k}: ASR={asr:.3f}  acuracia limpa (backdoor)={clean:.3f}")

Quanto **maior** o *k*, mais raro (seletivo) é o gatilho — a ASR continua alta e o vazamento sobre entradas limpas diminui.

## 4. Detecção textual (QASM como texto)

Tratamos o QASM como documento (BoW + bigramas) e treinamos um classificador linear para distinguir limpo vs. infectado. Demonstração mínima com os circuitos gerados acima.

In [ ]:
from qmltrojan.detect import run_detection

# Corpus de demonstracao: 3 circuitos-base (golden) + variantes infectadas de cada um.
# (compile_golden com pesos aleatorios e rapido; o importante aqui e a ESTRUTURA do QASM.)
texts, labels, groups = [], [], []
for base_seed in range(3):
    w = np.random.default_rng(base_seed).uniform(-np.pi, np.pi, len(trained.weights))
    base_golden = compile_golden(spec, w)
    texts.append(to_qasm3(base_golden)); labels.append(0); groups.append(base_seed)
    for i, gate in enumerate(["x", "y", "z", "t", "h"]):
        for mode in ["random", "idle", "controlled"]:
            inf, _ = apply_trojan(base_golden, TrojanSpec(gate=gate, mode=mode, seed=base_seed*10+i))
            texts.append(to_qasm3(inf)); labels.append(1); groups.append(base_seed)

# No experimento real usamos GroupKFold entre dezenas de modelos-base; aqui, um StratifiedKFold curto.
summary, cm = run_detection(texts, np.array(labels), np.array(groups),
                            "count", "logreg", "stratified", n_splits=3, seed=0)
print("recall:", round(summary["recall_mean"], 3),
      "| balanced acc:", round(summary["balanced_accuracy_mean"], 3),
      "| roc_auc:", round(summary["roc_auc_mean"], 3))
print("matriz de confusao (linhas=verdadeiro, colunas=previsto):")
print(cm)

## Conclusão

- Trojans de porta única deslocam a distribuição de saída de forma mensurável (TVD acima do *baseline*) e podem derrubar a acurácia.
- O backdoor com gatilho atinge ASR alta mantendo a acurácia limpa, sendo mais furtivo com gatilhos mais largos.
- A detecção textual é um ponto de partida, mas modesta — veja o `README.md` para a discussão honesta e as limitações.